In [37]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [38]:
df = pd.read_csv('diabetes_dataset.csv')
print(df.head)

<bound method NDFrame.head of        age  gender ethnicity education_level  income_level employment_status  \
0       58    Male     Asian      Highschool  Lower-Middle          Employed   
1       48  Female     White      Highschool        Middle          Employed   
2       60    Male  Hispanic      Highschool        Middle        Unemployed   
3       74  Female     Black      Highschool           Low           Retired   
4       46    Male     White        Graduate        Middle           Retired   
...    ...     ...       ...             ...           ...               ...   
99995   46    Male     Other        Graduate  Upper-Middle        Unemployed   
99996   41  Female     White        Graduate        Middle          Employed   
99997   57  Female     Black       No formal  Upper-Middle          Employed   
99998   47  Female     Black      Highschool  Lower-Middle           Retired   
99999   52  Female     White    Postgraduate        Middle          Employed   

      smo

In [ ]:
target = 'diagnosed_diabetes'
y = df[target]

In [40]:
numerical_features = [
    'age','alcohol_consumption_per_week','physical_activity_minutes_per_week',
    'diet_score','sleep_hours_per_day','screen_time_hours_per_day',
    'bmi','waist_to_hip_ratio','systolic_bp','diastolic_bp','heart_rate',
    'cholesterol_total','hdl_cholesterol','ldl_cholesterol','triglycerides',
    'glucose_fasting','glucose_postprandial','insulin_level','hba1c'
]

In [41]:
nominal_features = [
    'gender','ethnicity','employment_status','smoking_status',
    'family_history_diabetes','hypertension_history','cardiovascular_history','income_level'
]

In [42]:
ordinal_features = ['education_level']
education_order = ['No formal', 'Highschool', 'Graduate', 'Postgraduate']


In [43]:
X = df[numerical_features + nominal_features + ordinal_features]

In [44]:
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

In [45]:
nominal_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

In [46]:
ordinal_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ordinal', OrdinalEncoder(categories=[education_order]))
])


In [47]:
preprocessor = ColumnTransformer(transformers=[
    ('num', numerical_transformer, numerical_features),
    ('nom', nominal_transformer, nominal_features),
    ('ord', ordinal_transformer, ordinal_features)
])

In [48]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [49]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [50]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5)
}

In [51]:
# Train each model and evaluate
for name, model in models.items():
    model.fit(X_train_processed, y_train)
    y_pred = model.predict(X_test_processed)
    
    print(f"\n--- {name} ---")
    print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
    print("Classification Report:\n", classification_report(y_test, y_pred))


--- Logistic Regression ---
Accuracy: 0.8604
Confusion Matrix:
 [[ 6469  1531]
 [ 1261 10739]]
Classification Report:
               precision    recall  f1-score   support

           0       0.84      0.81      0.82      8000
           1       0.88      0.89      0.88     12000

    accuracy                           0.86     20000
   macro avg       0.86      0.85      0.85     20000
weighted avg       0.86      0.86      0.86     20000


--- Decision Tree ---
Accuracy: 0.8585
Confusion Matrix:
 [[ 6470  1530]
 [ 1300 10700]]
Classification Report:
               precision    recall  f1-score   support

           0       0.83      0.81      0.82      8000
           1       0.87      0.89      0.88     12000

    accuracy                           0.86     20000
   macro avg       0.85      0.85      0.85     20000
weighted avg       0.86      0.86      0.86     20000


--- KNN ---
Accuracy: 0.8125
Confusion Matrix:
 [[6302 1698]
 [2052 9948]]
Classification Report:
             

In [52]:
new_user = pd.DataFrame([{
    'age': 45,
    'alcohol_consumption_per_week': 3,
    'physical_activity_minutes_per_week': 120,
    'diet_score': 7,
    'sleep_hours_per_day': 7,
    'screen_time_hours_per_day': 5,
    'bmi': 28.5,
    'waist_to_hip_ratio': 0.95,
    'systolic_bp': 130,
    'diastolic_bp': 85,
    'heart_rate': 78,
    'cholesterol_total': 210,
    'hdl_cholesterol': 45,
    'ldl_cholesterol': 135,
    'triglycerides': 160,
    'glucose_fasting': 110,
    'glucose_postprandial': 160,
    'insulin_level': 18,
    'hba1c': 6.1,
    'gender': 'Male',
    'ethnicity': 'White',
    'employment_status': 'Employed',
    'smoking_status': 'Former',
    'family_history_diabetes': 1,
    'hypertension_history': 1,
    'cardiovascular_history': 0,
    'income_level': 'Middle',
    'education_level': 'Graduate'
}])

In [53]:
new_user_processed = preprocessor.transform(new_user)

In [54]:
# Predict with each model
for name, model in models.items():
    prediction = model.predict(new_user_processed)
    probability = model.predict_proba(new_user_processed)[:,1] if hasattr(model, "predict_proba") else None
    print(f"\n{name}:")
    print("Predicted Diabetes (0=No,1=Yes):", prediction[0])
    if probability is not None:
        print("Probability of Diabetes:", round(probability[0], 4))


Logistic Regression:
Predicted Diabetes (0=No,1=Yes): 0
Probability of Diabetes: 0.4522

Decision Tree:
Predicted Diabetes (0=No,1=Yes): 0
Probability of Diabetes: 0.0

KNN:
Predicted Diabetes (0=No,1=Yes): 0
Probability of Diabetes: 0.2


In [55]:
import joblib 
model = list(models.values())[0]
joblib.dump((preprocessor, model), 'diabetes_model.pkl')


['diabetes_model.pkl']